# 🏥 مراجع OCR الطبي — Medical OCR Reviewer

دفتر مُتكامل لمعاينة وتصحيح نتائج التعرف الضوئي على الحروف (OCR) للملاحظات الطبية المكتوبة بخط اليد.

### الميزات:
- تحميل ملفات PDF أو صور متعددة
- معالجة الصور مسبقًا (إزالة الخطوط + CLAHE)
- تقسيم الأسطر تلقائيًا
- التعرف باللغتين العربية والإنجليزية (EasyOCR)
- واجهة تفاعلية للمراجعة والتصحيح
- تصدير البيانات كـ ZIP (صور + تصنيفات) للتدريب

## الخطوة ١: تثبيت الحزم اللازمة

In [ ]:
# تثبيت حزم EasyOCR ومعالجة الصور وواجهة Gradio
!pip install -q easyocr opencv-python-headless gradio PyPDF2 Pillow

# التحقق من التثبيت
import easyocr
import cv2
import gradio as gr
from PIL import Image
import PyPDF2

print("✅ تمّ تثبيت جميع الحزم بنجاح")

## الخطوة ٢: تعريف صنف المراجع الطبي `MedicalOCRReviewer`

In [ ]:
import os
import io
import zipfile
import numpy as np
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import easyocr
import cv2
from PIL import Image
import PyPDF2


class MedicalOCRReviewer:
    """صنف مراجع التعرف الضوئي على الملاحظات الطبية.

    يوفّر:
    - تحميل ومعالجة ملفات PDF والصور
    - معالجة الصور مسبقًا (إزالة الخطوط + تحسين التباين)
    - تقسيم الصفحة إلى أسطر
    - التعرف على النصوص بالعربية والإنجليزية
    - واجهة للتنقل بين الأسطر والتصحيح
    - تصدير البيانات المُصحَّحة للتدريب
    """

    def __init__(self):
        # إنشاء قارئ EasyOCR للعربية والإنجليزية
        self.reader = easyocr.Reader(['ar', 'en'], gpu=True)

        # قائمة الصور المستخرجة
        self.images: List[np.ndarray] = []
        # قائمة الأسطر المقسومة (كل عنصر: صورة الأسطر)
        self.line_images: List[np.ndarray] = []
        # النصوص المتوقعة من OCR
        self.predictions: List[str] = []
        # النصوص المُصحَّحة من المستخدم
        self.corrections: List[str] = []
        # مؤشر السطر الحالي
        self.current_index: int = 0

    # ─────────────────────────────────────────────────────────
    # معالجة الصور مسبقًا
    # ─────────────────────────────────────────────────────────
    def preprocess_image(self, image: np.ndarray) -> np.ndarray:
        """معالجة الصورة مسبقًا لتحسين نتائج OCR.

        خطوات المعالجة:
            1. تحويل إلى تدرج رمادي
            2. إزالة الخطوط الأفقية والعمودية
            3. تطبيق CLAHE لتحسين التباين

        Args:
            image: صورة الإدخال (BGR أو RGB).

        Returns:
            صورة معالجة.
        """
        # التحقق من نوع الصورة وتحويلها
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image.copy()

        # عتبة ثنائية لفصل النص عن الخلفية
        _, binary = cv2.threshold(
            gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

        # ── إزالة الخطوط الأفقية ──
        # إنشاء عنصر هيكلي أفقي للكشف عن الخطوط
        horiz_kernel = cv2.getStructuringElement(
            cv2.MORPH_RECT, (40, 1)
        )
        # الكشف عن الخطوط الأفقية
        horiz_lines = cv2.morphologyEx(
            binary, cv2.MORPH_OPEN, horiz_kernel, iterations=2
        )
        # إزالة الخطوط الأفقية من الصورة
        no_horiz = cv2.subtract(binary, horiz_lines)

        # ── إزالة الخطوط العمودية ──
        # إنشاء عنصر هيكلي عمودي
        vert_kernel = cv2.getStructuringElement(
            cv2.MORPH_RECT, (1, 40)
        )
        # الكشف عن الخطوط العمودية
        vert_lines = cv2.morphologyEx(
            binary, cv2.MORPH_OPEN, vert_kernel, iterations=2
        )
        # إزالة الخطوط العمودية من الصورة
        cleaned = cv2.subtract(no_horiz, vert_lines)

        # ── تطبيق CLAHE (تحسين التباين التكيفي) ──
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(gray)

        # دمج النتيجة: تطبيق القناع المنظّف على الصورة المحسّنة
        final = cv2.bitwise_and(enhanced, enhanced, mask=cleaned)

        return final

    # ─────────────────────────────────────────────────────────
    # تقسيم الصفحة إلى أسطر
    # ─────────────────────────────────────────────────────────
    def segment_lines(self, image: np.ndarray) -> List[np.ndarray]:
        """تقسيم الصفحة إلى أسطر نصية.

        يعتمد على إسقاط أفقياً (Horizontal Projection)
        لتحديد مناطق الأسطر.

        Args:
            image: صورة الصفحة.

        Returns:
            قائمة بصور الأسطر.
        """
        # تحويل إلى رمادي إذا لزم
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image.copy()

        # عتبة ثنائية
        _, binary = cv2.threshold(
            gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

        # حساب الإسقاط الأفقي (مجموع البكسلات في كل صف)
        h_proj = np.sum(binary, axis=1)

        # تحديد مناطق الأسطر
        lines = []
        in_line = False
        start_y = 0
        min_line_height = 10  # أقل ارتفاع مقبول للسطر
        gap_threshold = 5     # عتبة الفاصل بين الأسطر

        for y, count in enumerate(h_proj):
            if count > 0 and not in_line:
                # بداية سطر جديد
                in_line = True
                start_y = y
            elif count == 0 and in_line:
                # نهاية سطر
                in_line = False
                line_height = y - start_y
                if line_height >= min_line_height:
                    lines.append((start_y, y))

        # إضافة آخر سطر إذا لم يُغلق
        if in_line:
            line_height = len(h_proj) - start_y
            if line_height >= min_line_height:
                lines.append((start_y, len(h_proj)))

        # قص الأسطر من الصورة الأصلية مع هوامش
        margin = 3
        line_images = []
        for i, (y1, y2) in enumerate(lines):
            # حساب هوامش آمنة
            top = max(0, y1 - margin)
            bottom = min(image.shape[0], y2 + margin)
            line_img = image[top:bottom, :]
            line_images.append(line_img)

        return line_images

    # ─────────────────────────────────────────────────────────
    # تحميل ومعالجة الملف
    # ─────────────────────────────────────────────────────────
    def load_and_process_file(self, file_path: str) -> Tuple[List[np.ndarray], List[str]]:
        """تحميل ملف PDF أو صورة ومعالجته.

        Args:
            file_path: مسار الملف.

        Returns:
            (قائمة صور الأسطر، قائمة النصوص المتوقعة)
        """
        self.images = []
        self.line_images = []
        self.predictions = []
        self.corrections = []
        self.current_index = 0

        # ── تحميل الصور من الملف ──
        if file_path.lower().endswith('.pdf'):
            # استخراج الصور من PDF
            with open(file_path, 'rb') as f:
                pdf_reader = PyPDF2.PdfReader(f)
                for page_num in range(len(pdf_reader.pages)):
                    page = pdf_reader.pages[page_num]
                    # تحويل الصفحة إلى صورة
                    # استخدام Pillow لتحويل PDF إلى صورة
                    # ملاحظة: PyPDF2 لا يدعم الاستخراج المباشر
                    # لذا نستخدم حلاً بديلاً
                    pass
            # إذا لم نتمكن من استخراج صور من PDF مباشرة
            # نبلغ المستخدم بتحويل PDF إلى صور مسبقًا
            raise NotImplementedError(
                "يرجى تحويل PDF إلى صور أولاً أو استخدام ملفات صور. "
                "يمكنك استخدام fitz (PyMuPDF) أو تحويل PDF إلى PNG."
            )
        else:
            # تحميل صورة واحدة
            img = cv2.imread(file_path)
            if img is None:
                raise ValueError(f"لم يتم تحميل الصورة: {file_path}")
            self.images.append(img)

        # ── معالجة كل صورة ──
        for page_img in self.images:
            # المعالجة المسبقة
            processed = self.preprocess_image(page_img)

            # تقسيم الأسطر
            lines = self.segment_lines(processed)
            self.line_images.extend(lines)

            # التعرف على النص في كل سطر
            for line_img in lines:
                # تحويل الصورة من BGR إلى RGB
                if len(line_img.shape) == 2:
                    rgb_line = cv2.cvtColor(line_img, cv2.COLOR_GRAY2RGB)
                else:
                    rgb_line = cv2.cvtColor(line_img, cv2.COLOR_BGR2RGB)

                # تشغيل OCR
                results = self.reader.readtext(rgb_line)

                # تجميع النصوص
                if results:
                    text = ' '.join([r[1] for r in results])
                else:
                    text = ""

                self.predictions.append(text)
                self.corrections.append(text)  # نسخة أولية للتصحيح

        print(f"✅ تمّ تحميل {len(self.line_images)} سطر من الملف")
        return self.line_images, self.predictions

    # ─────────────────────────────────────────────────────────
    # التنقل بين الأسطر
    # ─────────────────────────────────────────────────────────
    def get_current_line_ui(self) -> Tuple[np.ndarray, str, str, str]:
        """جلب بيانات السطر الحالي لعرضها في الواجهة.

        Returns:
            (صورة السطر، النص المتوقع، النص المُصحَّح، معلومات المؤشر)
        """
        if not self.line_images:
            # صورة فارغة إذا لم تُحمَّل بيانات
            blank = np.zeros((100, 400, 3), dtype=np.uint8)
            return blank, "", "", "لا توجد بيانات — حمّل ملفاً أولاً"

        idx = self.current_index
        line_img = self.line_images[idx]

        # تحويل إلى RGB للعرض
        if len(line_img.shape) == 2:
            display_img = cv2.cvtColor(line_img, cv2.COLOR_GRAY2RGB)
        else:
            display_img = cv2.cvtColor(line_img, cv2.COLOR_BGR2RGB)

        pred = self.predictions[idx]
        corr = self.corrections[idx]
        info = f"سطر {idx + 1} من {len(self.line_images)}"

        return display_img, pred, corr, info

    def save_and_next(self, corrected_text: str) -> Tuple[np.ndarray, str, str, str]:
        """حفظ التصحيح والانتقال للسطر التالي.

        Args:
            corrected_text: النص المُصحَّح.

        Returns:
            بيانات السطر التالي.
        """
        if not self.line_images:
            return self.get_current_line_ui()

        # حفظ التصحيح
        self.corrections[self.current_index] = corrected_text

        # الانتقال للسطر التالي
        if self.current_index < len(self.line_images) - 1:
            self.current_index += 1

        return self.get_current_line_ui()

    def save_and_prev(self, corrected_text: str) -> Tuple[np.ndarray, str, str, str]:
        """حفظ التصحيح والعودة للسطر السابق.

        Args:
            corrected_text: النص المُصحَّح.

        Returns:
            بيانات السطر السابق.
        """
        if not self.line_images:
            return self.get_current_line_ui()

        # حفظ التصحيح
        self.corrections[self.current_index] = corrected_text

        # العودة للسطر السابق
        if self.current_index > 0:
            self.current_index -= 1

        return self.get_current_line_ui()

    # ─────────────────────────────────────────────────────────
    # تصدير البيانات
    # ─────────────────────────────────────────────────────────
    def export_dataset(self, output_path: str = "medical_ocr_dataset.zip") -> str:
        """تصدير البيانات المُصحَّحة كملف ZIP للتدريب.

        هيكل الملف:
            medical_ocr_dataset.zip
            ├── images/
            │   ├── line_0001.png
            │   ├── line_0002.png
            │   └── ...
            └── labels.txt

        Args:
            output_path: مسار ملف ZIP الناتج.

        Returns:
            مسار ملف ZIP.
        """
        if not self.line_images:
            raise ValueError("لا توجد بيانات للتصدير. حمّل ملفاً أولاً.")

        # إنشاء ملف ZIP
        with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            # إنشاء ملف labels.txt
            labels_content = []

            for i, (line_img, corr_text) in enumerate(
                zip(self.line_images, self.corrections)
            ):
                # حفظ صورة السطر
                img_name = f"line_{i:04d}.png"
                img_path = f"images/{img_name}"

                # تحويل إلى RGB قبل الحفظ
                if len(line_img.shape) == 2:
                    rgb_img = cv2.cvtColor(line_img, cv2.COLOR_GRAY2RGB)
                else:
                    rgb_img = cv2.cvtColor(line_img, cv2.COLOR_BGR2RGB)

                # تشفير الصورة إلى PNG في الذاكرة
                success, img_encoded = cv2.imencode('.png', rgb_img)
                if success:
                    zf.writestr(img_path, img_encoded.tobytes())

                # إضافة التصنيف
                labels_content.append(f"{img_name}\t{corr_text}")

            # كتابة ملف التصنيفات
            zf.writestr("labels.txt", "\n".join(labels_content))

        file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"✅ تمّ تصدير {len(self.line_images)} سطر إلى: {output_path}")
        print(f"📦 حجم الملف: {file_size_mb:.2f} ميجابايت")

        return output_path


# ── إنشاء نسخة عامة من المراجع ──
reviewer = MedicalOCRReviewer()
print("✅ تمّ تهيئة مراجع OCR الطبي")

## الخطوة ٣: بناء واجهة Gradio التفاعلية

In [ ]:
import gradio as gr
import tempfile


def load_file_gradio(file_obj):
    """تحميل ملف في واجهة Gradio.

    Args:
        file_obj: كائن الملف المُحمَّل من Gradio.

    Returns:
        (صورة السطر، النص المتوقع، النص المُصحَّح، معلومات)
    """
    if file_obj is None:
        return reviewer.get_current_line_ui()

    file_path = file_obj.name if hasattr(file_obj, 'name') else file_obj
    try:
        reviewer.load_and_process_file(file_path)
    except NotImplementedError as e:
        blank = np.zeros((100, 600, 3), dtype=np.uint8)
        return (blank, "", "", str(e))
    except Exception as e:
        blank = np.zeros((100, 600, 3), dtype=np.uint8)
        return (blank, "", "", f"خطأ: {e}")

    return reviewer.get_current_line_ui()


def next_line_gradio(corrected_text):
    """الانتقال للسطر التالي مع حفظ التصحيح."""
    return reviewer.save_and_next(corrected_text)


def prev_line_gradio(corrected_text):
    """العودة للسطر السابق مع حفظ التصحيح."""
    return reviewer.save_and_prev(corrected_text)


def export_gradio():
    """تصدير البيانات من واجهة Gradio.

    Returns:
        مسار ملف ZIP.
    """
    output_path = "/content/medical_ocr_dataset.zip"
    try:
        reviewer.export_dataset(output_path)
        return output_path
    except Exception as e:
        return f"خطأ في التصدير: {e}"


# ── بناء الواجهة ──
with gr.Blocks(title="مراجع OCR الطبي", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏥 مراجع OCR الطبي — Medical OCR Reviewer")
    gr.Markdown("حمّل صورة أو PDF، ثم راجع وصحّح نصوص الأسطر.")

    with gr.Row():
        # عمود التحميل
        with gr.Column(scale=1):
            file_input = gr.File(
                label="📁 رفع الملف",
                file_types=[".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".pdf"],
            )
            load_btn = gr.Button("🔄 تحميل ومعالجة", variant="primary")

    # حالة العرض
    info_text = gr.Textbox(label="📌 المعلومات", interactive=False)

    with gr.Row():
        # عمود الصورة
        with gr.Column(scale=2):
            line_image = gr.Image(label="🖼️ صورة السطر", type="numpy")

        # عمود النصوص
        with gr.Column(scale=2):
            pred_text = gr.Textbox(
                label="🤖 النص المتوقع (OCR)",
                interactive=False,
                lines=3,
            )
            corr_text = gr.Textbox(
                label="✏️ النص المُصحَّح",
                interactive=True,
                lines=3,
                placeholder="أدخل التصحيح هنا...",
            )

    # أزرار التنقل
    with gr.Row():
        prev_btn = gr.Button("⬅️ السطر السابق")
        next_btn = gr.Button("➡️ السطر التالي")

    # التصدير
    with gr.Row():
        export_btn = gr.Button("📦 تصدير البيانات (ZIP)", variant="secondary")
    export_output = gr.File(label="📁 ملف ZIP المُصدَّر")

    # ── ربط الأحداث ──
    # تحميل الملف يعرض أول سطر
    load_btn.click(
        fn=load_file_gradio,
        inputs=[file_input],
        outputs=[line_image, pred_text, corr_text, info_text],
    )

    # التنقل بين الأسطر
    next_btn.click(
        fn=next_line_gradio,
        inputs=[corr_text],
        outputs=[line_image, pred_text, corr_text, info_text],
    )
    prev_btn.click(
        fn=prev_line_gradio,
        inputs=[corr_text],
        outputs=[line_image, pred_text, corr_text, info_text],
    )

    # التصدير
    export_btn.click(
        fn=export_gradio,
        inputs=[],
        outputs=[export_output],
    )


print("✅ تمّ بناء واجهة Gradio بنجاح")

## الخطوة ٤: تشغيل الواجهة

In [ ]:
# تشغيل واجهة Gradio مع رابط مشاركة عام
demo.launch(share=True, debug=True)